# 05 — Collaborative Pipeline

**Requires:** `GROQ_API_KEY` in `.env`  
**Optional:** `TAVILY_API_KEY` for live web search

---

## What's new here?

This is the **capstone** — a production-style multi-agent system with 5 agents  
that collaborate to produce a high-quality research brief on any topic.

```
 ┌─────────┐    ┌────────────┐    ┌─────────┐    ┌────────┐    ┌────────┐
 │ PLANNER │───▶│ RESEARCHER │───▶│ ANALYST │───▶│ WRITER │───▶│ CRITIC │
 └─────────┘    └────────────┘    └─────────┘    └────────┘    └───┬────┘
                                                                    │
                                                            ┌───────┴────────┐
                                                            │  needs revision │
                                                            │  score < 7     │
                                                            ▼                │
                                                         WRITER (loop) ──────┘
                                                            │
                                                         score ≥ 7
                                                            │
                                                           END
```

**What each agent does:**
| Agent | Role | Output |
|-------|------|--------|
| Planner | Breaks the topic into a research plan | `research_plan` |
| Researcher | Gathers information for each plan item | `raw_research` |
| Analyst | Identifies patterns, insights, contradictions | `analysis` |
| Writer | Writes a structured brief | `draft` |
| Critic | Scores the draft (1–10) and gives feedback | `feedback`, `quality_score` |

**Concepts covered:**
| Concept | What it means |
|---------|---------------|
| 5-node collaborative pipeline | Each agent builds on all previous work |
| Quality gate | Critic scores the draft; low score → revision loop |
| Revision loop | Writer improves the draft based on critic's feedback |
| Iteration limit | `revision_count` prevents infinite revision loops |
| Optional live search | Tavily gives the researcher real web results |

## Step 1 — Setup

In [ ]:
# %pip install langgraph langchain langchain-groq langchain-tavily python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file"

TAVILY_AVAILABLE = bool(os.getenv("TAVILY_API_KEY"))
print(f"Groq: ready")
print(f"Tavily (live search): {'enabled' if TAVILY_AVAILABLE else 'disabled — using mock data'}")

## Step 2 — Define the State

Each field maps to one agent's output.  
Later agents can see everything produced by earlier agents.

In [ ]:
from typing import TypedDict, List


class BriefState(TypedDict):
    # Input
    topic: str
    audience: str           # e.g. "business executives", "students"

    # Agent outputs (built up progressively)
    research_plan: str      # Planner
    raw_research: str       # Researcher
    analysis: str           # Analyst
    draft: str              # Writer
    feedback: str           # Critic

    # Quality control
    quality_score: int      # Critic's score (1-10)
    revision_count: int     # How many times Writer has revised


MAX_REVISIONS = 2  # Stop after this many revision attempts
QUALITY_THRESHOLD = 7  # Score needed to approve the draft

print(f"State defined. Quality threshold: {QUALITY_THRESHOLD}/10, Max revisions: {MAX_REVISIONS}")

## Step 3 — Set Up the LLM and Search Tool

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatGroq(model="llama3-8b-8192", temperature=0.6)


def search_web(query: str) -> str:
    """Search the web with Tavily, or use mock data if key not set."""
    if TAVILY_AVAILABLE:
        from langchain_tavily import TavilySearch
        searcher = TavilySearch(max_results=3)
        results = searcher.invoke(query)
        if isinstance(results, list):
            return "\n\n".join(
                f"Source: {r.get('url', 'unknown')}\n{r.get('content', '')}"
                for r in results[:3]
            )
        return str(results)
    else:
        # Fallback: let the LLM use its training knowledge
        return f"[Using LLM knowledge for: {query}]"


print("LLM and search ready.")

## Step 4 — Define the Five Agents

In [ ]:
# ─────────────────────────────────────────
# Agent 1: Planner
# ─────────────────────────────────────────
def planner_agent(state: BriefState) -> dict:
    """Break the topic into a structured research plan."""
    print("  [Planner] Creating research plan...")

    prompt = (
        f"Create a research plan for this topic: '{state['topic']}'\n"
        f"Target audience: {state['audience']}\n\n"
        f"List 4-5 specific research questions to answer.\n"
        f"For each question, note what kind of information is needed.\n"
        f"Be specific — these questions will guide the researcher."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  [Planner] Plan ready.")
    return {"research_plan": response.content}


# ─────────────────────────────────────────
# Agent 2: Researcher
# ─────────────────────────────────────────
def researcher_agent(state: BriefState) -> dict:
    """Gather information for each item in the research plan."""
    print("  [Researcher] Gathering information...")

    # Search for the main topic
    search_result = search_web(state["topic"])

    prompt = (
        f"You are a researcher. Answer each of these research questions about '{state['topic']}'.\n\n"
        f"RESEARCH PLAN:\n{state['research_plan']}\n\n"
        f"ADDITIONAL CONTEXT FROM SEARCH:\n{search_result}\n\n"
        f"Provide factual, detailed answers to each research question.\n"
        f"Include specific data, examples, and quotes where possible."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  [Researcher] Research complete.")
    return {"raw_research": response.content}


# ─────────────────────────────────────────
# Agent 3: Analyst
# ─────────────────────────────────────────
def analyst_agent(state: BriefState) -> dict:
    """Identify patterns, contradictions, and insights in the research."""
    print("  [Analyst] Analysing research...")

    prompt = (
        f"You are a senior analyst. Analyse this research on '{state['topic']}'.\n\n"
        f"RESEARCH:\n{state['raw_research']}\n\n"
        f"Your analysis must include:\n"
        f"1. Top 3 key insights (most important findings)\n"
        f"2. Surprising or counterintuitive findings (if any)\n"
        f"3. Gaps or limitations in the research\n"
        f"4. Implications for {state['audience']}\n"
        f"5. One concrete recommendation\n"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  [Analyst] Analysis complete.")
    return {"analysis": response.content}


# ─────────────────────────────────────────
# Agent 4: Writer
# ─────────────────────────────────────────
def writer_agent(state: BriefState) -> dict:
    """Write or revise the brief based on all available inputs."""
    is_revision = state["revision_count"] > 0
    action = "REVISE" if is_revision else "WRITE"
    print(f"  [Writer] {action} brief (revision #{state['revision_count']})...")

    revision_context = (
        f"\nPREVIOUS DRAFT:\n{state['draft']}\n\n"
        f"CRITIC FEEDBACK TO ADDRESS:\n{state['feedback']}\n\n"
        if is_revision else ""
    )

    prompt = (
        f"Write a professional research brief on '{state['topic']}' for {state['audience']}.\n\n"
        f"Use ALL of the following inputs:\n\n"
        f"RESEARCH:\n{state['raw_research'][:1500]}\n\n"
        f"ANALYSIS:\n{state['analysis']}\n\n"
        f"{revision_context}"
        f"FORMAT:\n"
        f"- Executive Summary (2-3 sentences)\n"
        f"- Key Findings (3-5 bullet points with data)\n"
        f"- Analysis & Implications\n"
        f"- Recommendation\n"
        f"- Conclusion\n\n"
        f"Length: 400-500 words. Tone: professional and clear."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [Writer] Draft complete ({len(response.content.split())} words).")
    return {
        "draft": response.content,
        "revision_count": state["revision_count"] + (1 if is_revision else 0),
    }


# ─────────────────────────────────────────
# Agent 5: Critic
# ─────────────────────────────────────────
def critic_agent(state: BriefState) -> dict:
    """Score the draft and provide specific improvement feedback."""
    print("  [Critic] Reviewing draft...")

    prompt = (
        f"You are a strict but fair editor. Review this research brief.\n\n"
        f"BRIEF:\n{state['draft']}\n\n"
        f"Evaluate on: accuracy, clarity, structure, completeness, usefulness for {state['audience']}.\n\n"
        f"Respond in EXACTLY this format:\n"
        f"SCORE: [number 1-10]\n"
        f"STRENGTHS: [2-3 things done well]\n"
        f"IMPROVEMENTS: [2-3 specific things to fix if score < 7]\n"
        f"VERDICT: [APPROVE if score >= 7, REVISE if score < 7]"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    feedback_text = response.content

    # Extract the score
    import re
    score_match = re.search(r'SCORE:\s*(\d+)', feedback_text)
    score = int(score_match.group(1)) if score_match else 6

    print(f"  [Critic] Score: {score}/10 — {'APPROVED' if score >= QUALITY_THRESHOLD else 'NEEDS REVISION'}")
    return {"feedback": feedback_text, "quality_score": score}


print("All 5 agents defined.")

## Step 5 — Define the Quality Gate

After the critic scores the draft, we decide:
- Score ≥ 7 → done, go to END  
- Score < 7 AND revisions remaining → loop back to Writer  
- Max revisions reached → accept the draft as-is

In [ ]:
def quality_gate(state: BriefState) -> str:
    """Route after critic: approve or send back for revision."""
    score = state["quality_score"]
    revisions = state["revision_count"]

    if score >= QUALITY_THRESHOLD:
        print(f"  [Quality Gate] Score {score} ≥ {QUALITY_THRESHOLD} → APPROVED")
        return "__end__"
    elif revisions >= MAX_REVISIONS:
        print(f"  [Quality Gate] Max revisions reached → accepting draft")
        return "__end__"
    else:
        print(f"  [Quality Gate] Score {score} < {QUALITY_THRESHOLD} → REVISING")
        return "writer"


print("Quality gate defined.")

## Step 6 — Build the Full Graph

```
START → planner → researcher → analyst → writer → critic
                                                      │
                                           ┌──────────┴──────────┐
                                     (score ≥ 7)           (score < 7)
                                           │                     │
                                          END              writer (loop)
```

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(BriefState)

# Register all 5 agents
graph.add_node("planner",    planner_agent)
graph.add_node("researcher", researcher_agent)
graph.add_node("analyst",    analyst_agent)
graph.add_node("writer",     writer_agent)
graph.add_node("critic",     critic_agent)

# Main pipeline
graph.add_edge(START, "planner")
graph.add_edge("planner",    "researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst",    "writer")
graph.add_edge("writer",     "critic")

# Quality gate: critic decides → END or loop back to writer
graph.add_conditional_edges(
    "critic",
    quality_gate,
    {
        "writer": "writer",
        "__end__": END,
    },
)

pipeline = graph.compile()
print("Pipeline compiled.")

## Step 7 — Run the Full Pipeline

In [ ]:
initial_state: BriefState = {
    "topic": "The impact of generative AI on creative industries",
    "audience": "business executives and creative directors",
    "research_plan": "",
    "raw_research": "",
    "analysis": "",
    "draft": "",
    "feedback": "",
    "quality_score": 0,
    "revision_count": 0,
}

print(f"Topic: {initial_state['topic']}")
print(f"Audience: {initial_state['audience']}")
print("=" * 60)
print()

result = pipeline.invoke(initial_state)

In [ ]:
print("\n" + "=" * 60)
print("FINAL RESEARCH BRIEF")
print("=" * 60)
print(result["draft"])

print("\n" + "=" * 60)
print("QUALITY METRICS")
print("=" * 60)
print(f"Final score:      {result['quality_score']}/10")
print(f"Revisions made:   {result['revision_count']}")
print(f"\nCritic feedback:\n{result['feedback']}")

## Step 8 — Inspect the Full Pipeline

See what each agent contributed.

In [ ]:
sections = [
    ("RESEARCH PLAN (Planner)",     result["research_plan"]),
    ("RAW RESEARCH (Researcher)",   result["raw_research"][:600] + "..."),
    ("ANALYSIS (Analyst)",          result["analysis"]),
]

for title, content in sections:
    print(f"\n{'=' * 60}")
    print(title)
    print("-" * 60)
    print(content)

## Step 9 — Reusable Function

In [ ]:
def create_brief(topic: str, audience: str = "general readers") -> dict:
    """Run the full 5-agent pipeline and return the results."""
    state: BriefState = {
        "topic": topic,
        "audience": audience,
        "research_plan": "",
        "raw_research": "",
        "analysis": "",
        "draft": "",
        "feedback": "",
        "quality_score": 0,
        "revision_count": 0,
    }
    result = pipeline.invoke(state)
    return {
        "brief": result["draft"],
        "score": result["quality_score"],
        "revisions": result["revision_count"],
    }


# Try your own topic
output = create_brief(
    topic="Electric vehicles and the future of urban transportation",
    audience="city planners and policymakers",
)

print(output["brief"])
print(f"\nScore: {output['score']}/10 | Revisions: {output['revisions']}")

## Step 10 — Visualise

In [ ]:
from IPython.display import Image, display

try:
    display(Image(pipeline.get_graph().draw_mermaid_png()))
except Exception:
    print(pipeline.get_graph().draw_mermaid())

## Summary

| Agent | Role | Key state field |
|-------|------|----------------|
| Planner | Structures the problem | `research_plan` |
| Researcher | Gathers raw information | `raw_research` |
| Analyst | Extracts insights | `analysis` |
| Writer | Produces the deliverable | `draft` |
| Critic | Enforces quality | `quality_score`, `feedback` |

**What makes this a "real" multi-agent system:**
- Each agent has a **single, focused responsibility**
- Agents build on each other's work through **shared state**
- The **quality gate** makes the system self-correcting
- The **iteration limit** prevents infinite loops in production

---

## Series Complete!

| Notebook | Key concept |
|----------|-------------|
| 01 — Agent Basics | State, nodes, loops, tool routing |
| 02 — Tool Calling | LLM selects tools with ReAct pattern |
| 03 — Two-Agent System | State handoff between agents |
| 04 — Supervisor Pattern | Dynamic routing by a manager agent |
| 05 — Collaborative Pipeline | Full production system with quality gate |

**Where to go next:**
- **Memory** — persist state across sessions with `MemorySaver`
- **Human-in-the-loop** — use `interrupt()` to pause for approval
- **Parallel agents** — use the `Send` API to fan out work
- **LangGraph Platform** — deploy your multi-agent system as an API